# 03 - Model Training
Trains and saves the Deployment Random Forest pipeline and full multi-class model.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.makedirs('../outputs/reports', exist_ok=True)
os.makedirs('../models', exist_ok=True)
from src.data.generate_dataset import generate_network_traffic, generate_phishing_dataset, generate_malware_dataset
from src.models.train import train_deployment_model, train_logistic_regression, compute_metrics
from src.preprocessing.pipeline import prepare_training_data
import pandas as pd
print('Imports OK')

Imports OK


In [2]:
raw = generate_network_traffic(num_rows=2000, random_seed=42)
prep = prepare_training_data(raw, test_size=0.2, random_state=42)

deploy_model, deploy_metrics = train_deployment_model(
    prep['X_deploy_train'], prep['y_binary_train'],
    prep['X_deploy_test'], prep['y_binary_test']
)
print('Deployment Model (3-Feature RF) Metrics:')
print(deploy_metrics)

[2026-10-16 16:48:22] [INFO] [threat_detection]: Generating 2000 synthetic network traffic records (Seed=42)...


[2026-10-16 16:48:23] [INFO] [threat_detection]: Generated network dataset with shape (2004, 17).


[2026-10-16 16:48:23] [INFO] [threat_detection]: Cleaning raw data. Initial shape: (2004, 17)


[2026-10-16 16:48:23] [INFO] [threat_detection]: Removed 4 duplicate records.


[2026-10-16 16:48:23] [INFO] [threat_detection]: Cleaning completed. Cleaned shape: (2000, 18)


[2026-10-16 16:48:23] [INFO] [threat_detection]: Training Deployment Model (3 features: packet_size, failed_logins, request_frequency)...


[2026-10-16 16:48:23] [INFO] [threat_detection]: Deployment Model Test Metrics: {'accuracy': 0.985, 'precision': 0.9851, 'recall': 0.985, 'f1': 0.9849, 'roc_auc': 0.9861}


Deployment Model (3-Feature RF) Metrics:
{'accuracy': 0.985, 'precision': 0.9851, 'recall': 0.985, 'f1': 0.9849, 'roc_auc': 0.9861}


In [3]:
lr_model, lr_metrics = train_logistic_regression(
    prep['X_full_train'], prep['y_binary_train'],
    prep['X_full_test'], prep['y_binary_test'],
    prep['numeric_cols'], prep['categorical_cols']
)
print('Logistic Regression Baseline Metrics:')
print(lr_metrics)

[2026-10-16 16:48:23] [INFO] [threat_detection]: Training baseline Logistic Regression model...


[2026-10-16 16:48:25] [INFO] [threat_detection]: Logistic Regression Test Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0}


Logistic Regression Baseline Metrics:
{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0}


In [4]:
results = pd.DataFrame([
    {'Model': 'Deployment RF (3 features)', **deploy_metrics},
    {'Model': 'Logistic Regression Baseline', **lr_metrics}
])
print('\nModel Comparison:')
results


Model Comparison:


,Model,accuracy,precision,recall,f1,roc_auc
0,Deployment RF (3 features),0.985,0.9851,0.985,0.9849,0.9861
1,Logistic Regression Baseline,1.000,1.0000,1.000,1.0000,1.0000
